# 🌍 Google Translate: The Encoder-Decoder Transformer

**Goal:** Build the core architectural pieces of Google Translate from scratch — BPE tokenization, encoder self-attention, decoder causal masking, and cross-attention.

**Vibe:** You're a bilingual wizard who reads an entire French novel, fully understands it, and then writes the English version. One word at a time — but always peeking back at the French original. 🧙‍♂️📚

---

## 🤔 Why Encoder-Decoder for Translation?

### The French Novel Analogy 📖

Imagine your job is to translate a French novel into English.

**Bad approach (what a robot would naively do):**
> Read word 1 in French → write its English equivalent → read word 2 → write its English equivalent → ...

This fails spectacularly. "Je l'aime" literally means "I it-love" — you need to know what "l'" refers to before you even start!

**Good approach (what a human translator does):**
> 1. 📖 **READ** the full French paragraph. Understand the meaning completely.
> 2. ✍️ **WRITE** the English version word by word — but keep the French paragraph open and glance back whenever you need it.

This is **exactly** the encoder-decoder architecture:

```
ENCODER (Step 1 — READ)                DECODER (Step 2 — WRITE)
═══════════════════════                ════════════════════════

"Le chat est sur le tapis"             "The cat is on the mat"
         │                                       ^
         ▼                                       │
 ┌───────────────┐          memory          ┌──────────────┐
 │   ENCODER     │  ──── rich vectors ────► │   DECODER    │
 │               │    (each source token    │              │
 │  Reads ALL    │     gets a context-      │  Generates   │
 │  tokens at    │     aware embedding)     │  one token   │
 │  once ✅      │                          │  at a time   │
 └───────────────┘                          └──────────────┘

 🔑 Key: Encoder sees EVERYTHING (bidirectional)
         Decoder generates LEFT-TO-RIGHT (causal)
         They talk via CROSS-ATTENTION (the bridge 🌉)
```

### Why NOT just use GPT (decoder-only)?

GPT processes tokens left-to-right. When it's generating "The" (the first English word), it hasn't even "seen" the end of the French sentence yet!

But in French → English translation, word order changes completely:
- French: "C'est le chien que j'aime" ("It is the dog that I love")
- You need to see the ENTIRE source sentence to know where to start the English.

The encoder fixes this: it processes the entire source first, then hands a rich understanding to the decoder.

| Model Type | Sees Source | Generates | Best For |
|---|---|---|---|
| **Encoder-only (BERT)** | Full bidirectional | ❌ Can't generate | Classification, NER |
| **Decoder-only (GPT)** | Left-to-right only | ✅ Yes | Chat, completion |
| **Encoder-Decoder (T5)** | Full source ✅ | ✅ Yes | Translation, summarization |

> 🎯 **Interview gold:** "We use encoder-decoder because translation is a sequence-to-sequence task where the source word order differs from the target. The encoder gives bidirectional context over the full source; the decoder generates autoregressively while attending back to the encoder via cross-attention."


## ✂️ Byte-Pair Encoding (BPE): The Subword Magic

### The Problem: Vocabulary Explosion

If you use whole words as tokens, you need a gigantic vocabulary:
- English: ~170,000 words
- Add French, German, Spanish, Chinese, Arabic, Japanese... → millions of tokens!
- And you STILL can't handle:
  - New words: "COVID-19", "selfie", "YOLO"
  - Typos: "teh", "recieve"
  - Rare words: "phytoplankton", "schadenfreude"

### The BPE Solution: Break Words into Pieces!

```
WORD TOKENIZATION (bad for rare words):
  "unhappiness" → [<UNK>]   (never seen this word!)

CHARACTER TOKENIZATION (too many tokens):
  "unhappiness" → ['u','n','h','a','p','p','i','n','e','s','s']  (11 tokens!)

BPE TOKENIZATION (Goldilocks 🐻):
  "unhappiness" → ['un', 'happi', 'ness']  (just 3 tokens, still meaningful!)
```

### How BPE Works: The Merging Algorithm

BPE learns by **iteratively merging the most frequent character pair** in the training corpus:

```
START: Character-level vocabulary
Corpus: cat(5x), cats(3x), dog(4x), dogs(2x)

Characters: c-a-t</w>, c-a-t-s</w>, d-o-g</w>, d-o-g-s</w>
                                                    (</w> = end of word)

ITERATION 1: Most frequent pair?
  Count pairs: (c,a)=8, (a,t)=8, (t,</w>)=5, (t,s)=3, (d,o)=6, (o,g)=6...
  Tie between (c,a) and (a,t) — merge (c,a) → "ca"
  New vocab now has "ca" as a unit!

ITERATION 2: Most frequent pair?
  (ca,t)=8, (t,</w>)=5, (d,o)=6, (o,g)=6...
  Merge (ca,t) → "cat"!

ITERATION 3: Most frequent pair?
  (d,o)=6 — merge (d,o) → "do"

ITERATION 4: Merge (do,g) → "dog"!

RESULT:
  "cats" → ["cat", "s</w>"]     (it learned that "cats" = "cat" + plural!)
  "dogs" → ["dog", "s</w>"]     (same pattern — genius! 🧠)
```

### Why is this great for translation?

- **Shared subwords across languages:** French "chat" and English "cat" share the "cat" subword concept
- **Morphology for free:** Prefixes (un-, re-, pre-) and suffixes (-ing, -tion, -ness) become reusable tokens
- **Compact:** ~32,000-64,000 subword tokens cover most languages
- **No UNK tokens:** Any string can be tokenized (worst case: individual characters)


In [ ]:
"""
✂️ BPE from Scratch!

We'll implement Byte-Pair Encoding step by step using the canonical
cat/cats/dog/dogs example from the book. Watch each merge happen live!
"""

from collections import defaultdict, Counter
import copy

# ═══════════════════════════════════════════════════════
# STEP 1: Define our mini corpus with word frequencies
# ═══════════════════════════════════════════════════════
# Format: {word: frequency} — the word is pre-split into characters
# We use '</w>' to mark end-of-word (helps distinguish "cat" from "cats")

def get_initial_vocab(corpus):
    """Split words into characters, add </w> end-of-word marker."""
    vocab = {}
    for word, freq in corpus.items():
        # Split into list of characters, add end-of-word
        chars = tuple(list(word) + ['</w>'])
        vocab[chars] = freq
    return vocab


def get_pair_frequencies(vocab):
    """Count how often each adjacent pair of tokens appears in the vocab."""
    pair_freq = defaultdict(int)
    for word_tokens, freq in vocab.items():
        for i in range(len(word_tokens) - 1):
            pair = (word_tokens[i], word_tokens[i+1])
            pair_freq[pair] += freq
    return pair_freq


def merge_pair(vocab, best_pair):
    """Merge the best pair into a new token everywhere it appears."""
    new_vocab = {}
    merged_token = ''.join(best_pair)  # e.g., ('c', 'a') → 'ca'
    for word_tokens, freq in vocab.items():
        new_tokens = []
        i = 0
        while i < len(word_tokens):
            # If current + next tokens form the best pair, merge them
            if (i < len(word_tokens) - 1 and
                    word_tokens[i] == best_pair[0] and
                    word_tokens[i+1] == best_pair[1]):
                new_tokens.append(merged_token)
                i += 2  # Skip both tokens
            else:
                new_tokens.append(word_tokens[i])
                i += 1
        new_vocab[tuple(new_tokens)] = freq
    return new_vocab


def get_all_tokens(vocab):
    """Get the set of all unique tokens currently in the vocabulary."""
    tokens = set()
    for word_tokens in vocab.keys():
        for t in word_tokens:
            tokens.add(t)
    return sorted(tokens)


# ═══════════════════════════════════════════════════════
# STEP 2: Run BPE for N iterations
# ═══════════════════════════════════════════════════════

# Our mini corpus from the book: word → frequency
corpus = {
    'cat':  5,
    'cats': 3,
    'dog':  4,
    'dogs': 2,
}

print("═" * 60)
print("  🐱🐶 BPE From Scratch: cat/cats/dog/dogs")
print("═" * 60)

vocab = get_initial_vocab(corpus)
print(f"\n📌 INITIAL VOCABULARY (characters only):")
for word_tokens, freq in vocab.items():
    print(f"   {''.join(c for c in word_tokens if c != '</w>')} ({freq}x) → {list(word_tokens)}")
print(f"\n   All tokens: {get_all_tokens(vocab)}")
print(f"   Vocab size: {len(get_all_tokens(vocab))} tokens")

# Run 6 BPE merge iterations
NUM_MERGES = 6
merge_rules = []  # We'll save these to use for tokenizing new words later

for iteration in range(1, NUM_MERGES + 1):
    pair_freq = get_pair_frequencies(vocab)
    if not pair_freq:
        print(f"\n⚠️  No more pairs to merge!")
        break

    # Find the most frequent pair (ties broken alphabetically for determinism)
    best_pair = max(pair_freq, key=lambda p: (pair_freq[p], p))
    best_freq = pair_freq[best_pair]

    # Perform the merge
    vocab = merge_pair(vocab, best_pair)
    merge_rules.append(best_pair)
    merged_token = ''.join(best_pair)

    print(f"\n{'─' * 60}")
    print(f"🔀 ITERATION {iteration}: Merge {best_pair} → '{merged_token}'  (freq={best_freq})")
    print(f"{'─' * 60}")
    print("   Current tokenization:")
    for word_tokens, freq in vocab.items():
        original = ''.join(c for c in word_tokens if c != '</w>')
        print(f"   {'':4}{original} ({freq}x) → {list(word_tokens)}")
    print(f"\n   All tokens: {get_all_tokens(vocab)}")
    print(f"   Vocab size: {len(get_all_tokens(vocab))} tokens")

print(f"\n{'═' * 60}")
print("✅ LEARNED MERGE RULES (in order):")
for i, rule in enumerate(merge_rules, 1):
    print(f"   {i}. {rule[0]} + {rule[1]} → {''.join(rule)}")
print(f"{'═' * 60}")

In [ ]:
"""
🌐 Build a Multilingual BPE Vocabulary and Tokenize Sentence Pairs

Now we'll extend to an English + French mini-corpus and show how
BPE handles BOTH languages with a SHARED vocabulary!
"""

# ═══════════════════════════════════════════════════════
# A small but realistic multilingual corpus
# ═══════════════════════════════════════════════════════
multilingual_corpus = {
    # English
    'the': 50, 'cat': 12, 'cats': 5, 'sat': 8, 'on': 20,
    'mat': 6, 'dog': 10, 'dogs': 3, 'run': 9, 'runs': 4,
    'running': 3, 'walked': 5, 'walking': 4,
    # French
    'le': 45, 'la': 40, 'chat': 11, 'chats': 4, 'est': 15,
    'sur': 8, 'tapis': 5, 'chien': 9, 'chiens': 3,
    'court': 7, 'marche': 6, 'marchait': 4,
}

print("═" * 65)
print("  🌍 Multilingual BPE: English + French")
print("═" * 65)

# Build BPE vocab with more merges
vocab_multi = get_initial_vocab(multilingual_corpus)
print(f"\n📊 Initial character vocab size: {len(get_all_tokens(vocab_multi))} tokens")

multilingual_merge_rules = []
NUM_MERGES_MULTI = 20

for iteration in range(1, NUM_MERGES_MULTI + 1):
    pair_freq = get_pair_frequencies(vocab_multi)
    if not pair_freq:
        break
    best_pair = max(pair_freq, key=lambda p: (pair_freq[p], p))
    vocab_multi = merge_pair(vocab_multi, best_pair)
    multilingual_merge_rules.append(best_pair)

final_tokens = get_all_tokens(vocab_multi)
print(f"After {NUM_MERGES_MULTI} merges, vocab size: {len(final_tokens)} tokens")
print(f"\n📜 Final token vocabulary:")
# Display tokens neatly in columns
for i, token in enumerate(final_tokens):
    end = '  ' if (i + 1) % 8 != 0 else '\n'
    print(f"'{token}'", end=end)
print()

# ═══════════════════════════════════════════════════════
# Tokenize new sentences using our learned BPE rules
# ═══════════════════════════════════════════════════════

def bpe_tokenize(word, merge_rules):
    """Tokenize a word using learned BPE merge rules."""
    tokens = list(word) + ['</w>']
    for rule in merge_rules:
        merged = ''.join(rule)
        new_tokens = []
        i = 0
        while i < len(tokens):
            if (i < len(tokens) - 1 and
                    tokens[i] == rule[0] and
                    tokens[i+1] == rule[1]):
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    # Remove </w> from display but show it happened
    return [t.replace('</w>', '') if t != '</w>' else '▁' for t in tokens]


def tokenize_sentence(sentence, merge_rules):
    """Tokenize a full sentence."""
    all_tokens = []
    for word in sentence.lower().split():
        word_tokens = bpe_tokenize(word.strip('.,!?'), merge_rules)
        all_tokens.extend(word_tokens)
    return all_tokens


# Test on English-French sentence pairs
sentence_pairs = [
    ("The cat sat on the mat", "Le chat est sur le tapis"),
    ("The dogs are running", "Les chiens courent"),
    ("Walking is healthy", "Marcher est sain"),
]

print("\n" + "═" * 65)
print("  📝 Tokenizing English-French Sentence Pairs")
print("═" * 65)

for en, fr in sentence_pairs:
    en_tokens = tokenize_sentence(en, multilingual_merge_rules)
    fr_tokens = tokenize_sentence(fr, multilingual_merge_rules)
    print(f"\n🇬🇧 EN: '{en}'")
    print(f"       Tokens: {en_tokens} ({len(en_tokens)} tokens)")
    print(f"🇫🇷 FR: '{fr}'")
    print(f"       Tokens: {fr_tokens} ({len(fr_tokens)} tokens)")

# Show cross-lingual subword sharing
print("\n" + "═" * 65)
print("  🌉 Cross-Lingual Subword Sharing")
print("═" * 65)
shared_examples = [
    ('cat', 'chat'),
    ('mat', 'tapis'),
    ('running', 'marche'),
]
for en_word, fr_word in shared_examples:
    en_t = bpe_tokenize(en_word, multilingual_merge_rules)
    fr_t = bpe_tokenize(fr_word, multilingual_merge_rules)
    print(f"   '{en_word}' → {en_t}")
    print(f"   '{fr_word}' → {fr_t}")
    common = set(en_t) & set(fr_t) - {'▁'}
    if common:
        print(f"   ✨ Shared subwords: {common}")
    print()

print("💡 KEY INSIGHT: BPE naturally discovers morphological units")
print("   (prefixes, suffixes, stems) that work across languages!")

## 🔍 Encoder Architecture: Bidirectional Self-Attention

The encoder's superpower is that **every token can attend to every other token** — it sees the whole sentence at once.

```
ENCODER SELF-ATTENTION (Bidirectional — sees EVERYTHING)
═════════════════════════════════════════════════════════

         The   cat   sat   on   the   mat
   The  [ ✅    ✅    ✅    ✅    ✅    ✅ ]
   cat  [ ✅    ✅    ✅    ✅    ✅    ✅ ]
   sat  [ ✅    ✅    ✅    ✅    ✅    ✅ ]
   on   [ ✅    ✅    ✅    ✅    ✅    ✅ ]
   the  [ ✅    ✅    ✅    ✅    ✅    ✅ ]
   mat  [ ✅    ✅    ✅    ✅    ✅    ✅ ]

All green! 🟢 Every word can look at every other word.
```

Why does this matter?

- **"bank"** in "river bank" vs "money bank" — the encoder sees the whole sentence and knows which meaning applies
- **"The cat** that chased the dog **sat** down" — "sat" can look all the way back to "cat" across many words
- **Verb agreement** — "The cats [that lived in my house all winter] **are** hungry" (plural because of "cats", not "house")

### The Mechanics: Q, K, V

```
For each token, we compute three vectors:

  Q (Query)  = "What am I looking for?"
  K (Key)    = "What do I contain?"
  V (Value)  = "What information do I give if you attend to me?"

Attention score between token i and token j:
  score(i,j) = softmax(Q_i · K_j / √d_k)

Output for token i:
  out_i = Σ_j score(i,j) * V_j
          ↑ weighted average of ALL value vectors!

For the ENCODER: no masking — all scores are valid.
```


In [ ]:
"""
🎭 Encoder vs Decoder Attention Masks

We'll implement both attention types and visualize them as heatmaps.
The KEY difference: encoder = see everything, decoder = causal masking.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

np.random.seed(42)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Core attention mechanism.
    Q, K, V: [seq_len, d_k]
    mask: [seq_len, seq_len] — 1 = attend, 0 = mask out
    Returns: (output, attention_weights)
    """
    d_k = Q.shape[-1]
    # [seq_len, seq_len] — raw attention scores
    scores = Q @ K.T / np.sqrt(d_k)

    if mask is not None:
        # Fill masked positions with -inf so softmax → 0
        scores = np.where(mask == 0, -1e9, scores)

    # Softmax over the last dimension (over keys)
    # Numerically stable softmax
    scores_shifted = scores - scores.max(axis=-1, keepdims=True)
    exp_scores = np.exp(scores_shifted)
    attn_weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)

    output = attn_weights @ V
    return output, attn_weights


def make_causal_mask(seq_len):
    """Lower triangular mask — token i can only attend to tokens 0..i."""
    return np.tril(np.ones((seq_len, seq_len)))


def make_encoder_mask(seq_len):
    """All-ones mask — every token attends to every other."""
    return np.ones((seq_len, seq_len))


# ═══════════════════════════════════════════════════════
# Example sentences
# ═══════════════════════════════════════════════════════
src_tokens = ['The', 'cat', 'sat', 'on', 'the', 'mat']
tgt_tokens = ['Le', 'chat', 's\'est', 'assis', 'sur', 'le', 'tapis']

src_len = len(src_tokens)
tgt_len = len(tgt_tokens)
d_k = 8  # Small dimension for illustration

# Simulate Q, K, V matrices (in real life these come from learned linear layers)
Q_enc = np.random.randn(src_len, d_k)
K_enc = np.random.randn(src_len, d_k)
V_enc = np.random.randn(src_len, d_k)

Q_dec = np.random.randn(tgt_len, d_k)
K_dec = np.random.randn(tgt_len, d_k)
V_dec = np.random.randn(tgt_len, d_k)

# ═══════════════════════════════════════════════════════
# Run attention with both mask types
# ═══════════════════════════════════════════════════════
enc_mask = make_encoder_mask(src_len)
dec_mask = make_causal_mask(tgt_len)

_, enc_attn = scaled_dot_product_attention(Q_enc, K_enc, V_enc, enc_mask)
_, dec_attn = scaled_dot_product_attention(Q_dec, K_dec, V_dec, dec_mask)

# ═══════════════════════════════════════════════════════
# Visualize: mask structure + attention weights
# ═══════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# ─── Plot 1: Encoder mask (all green)
ax = axes[0, 0]
im1 = ax.imshow(enc_mask, cmap='Greens', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(src_len))
ax.set_yticks(range(src_len))
ax.set_xticklabels(src_tokens, fontsize=11)
ax.set_yticklabels(src_tokens, fontsize=11)
ax.set_title('🟢 Encoder Mask\n(Bidirectional — ALL tokens attend to ALL)', fontsize=13, fontweight='bold')
ax.set_xlabel('Key (Which token am I attending TO?)', fontsize=10)
ax.set_ylabel('Query (Which token is ASKING?)', fontsize=10)
for i in range(src_len):
    for j in range(src_len):
        ax.text(j, i, '✅', ha='center', va='center', fontsize=9)
ax.set_xticks(np.arange(src_len) - 0.5, minor=True)
ax.set_yticks(np.arange(src_len) - 0.5, minor=True)
ax.grid(which='minor', color='white', linewidth=2)

# ─── Plot 2: Encoder attention weights (softmax output)
ax = axes[0, 1]
im2 = ax.imshow(enc_attn, cmap='Blues', vmin=0, vmax=enc_attn.max(), aspect='auto')
ax.set_xticks(range(src_len))
ax.set_yticks(range(src_len))
ax.set_xticklabels(src_tokens, fontsize=11)
ax.set_yticklabels(src_tokens, fontsize=11)
ax.set_title('🔵 Encoder Attention Weights\n(Softmax output — each row sums to 1.0)', fontsize=13, fontweight='bold')
ax.set_xlabel('Attending TO', fontsize=10)
ax.set_ylabel('Attending FROM', fontsize=10)
for i in range(src_len):
    for j in range(src_len):
        ax.text(j, i, f'{enc_attn[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='white' if enc_attn[i,j] > 0.2 else 'black')
plt.colorbar(im2, ax=ax)

# ─── Plot 3: Decoder causal mask (lower triangular)
ax = axes[1, 0]
colors = ['#ffeeee', '#d4edda']  # light red for blocked, light green for allowed
cmap_mask = mcolors.ListedColormap(colors)
im3 = ax.imshow(dec_mask, cmap=cmap_mask, vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(tgt_len))
ax.set_yticks(range(tgt_len))
ax.set_xticklabels(tgt_tokens, fontsize=9, rotation=20)
ax.set_yticklabels(tgt_tokens, fontsize=9)
ax.set_title('🔴 Decoder Causal Mask\n(Can only look at PAST tokens — no peeking!)', fontsize=13, fontweight='bold')
ax.set_xlabel('Key (Which token am I attending TO?)', fontsize=10)
ax.set_ylabel('Query (Which token is ASKING?)', fontsize=10)
for i in range(tgt_len):
    for j in range(tgt_len):
        symbol = '✅' if dec_mask[i, j] == 1 else '🚫'
        ax.text(j, i, symbol, ha='center', va='center', fontsize=8)
ax.set_xticks(np.arange(tgt_len) - 0.5, minor=True)
ax.set_yticks(np.arange(tgt_len) - 0.5, minor=True)
ax.grid(which='minor', color='white', linewidth=1.5)

# ─── Plot 4: Decoder attention weights (masked softmax)
ax = axes[1, 1]
im4 = ax.imshow(dec_attn, cmap='Reds', vmin=0, vmax=dec_attn.max(), aspect='auto')
ax.set_xticks(range(tgt_len))
ax.set_yticks(range(tgt_len))
ax.set_xticklabels(tgt_tokens, fontsize=9, rotation=20)
ax.set_yticklabels(tgt_tokens, fontsize=9)
ax.set_title('🔴 Decoder Attention Weights\n(Future positions are 0.00 — fully blocked)', fontsize=13, fontweight='bold')
ax.set_xlabel('Attending TO', fontsize=10)
ax.set_ylabel('Attending FROM', fontsize=10)
for i in range(tgt_len):
    for j in range(tgt_len):
        ax.text(j, i, f'{dec_attn[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if dec_attn[i,j] > 0.3 else 'black')
plt.colorbar(im4, ax=ax)

plt.suptitle('Encoder (Bidirectional) vs Decoder (Causal) Attention',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n🎯 KEY OBSERVATIONS:")
print("  Encoder mask: ALL 1s — every token attends everywhere")
print("  Decoder mask: Lower-triangular — row i can only see columns 0..i")
print("  The weights row sums to 1.0 (softmax normalization)")
print("  Decoder future positions: 0.00 — the -inf masking worked! 🎉")

# Quick sanity checks
print("\n✅ Sanity checks:")
print(f"  Encoder row sums (should all be ~1.0): {enc_attn.sum(axis=1).round(4)}")
print(f"  Decoder row sums (should all be ~1.0): {dec_attn.sum(axis=1).round(4)}")
print(f"  Decoder upper-triangle (should all be ~0): "
      f"{dec_attn[np.triu_indices(tgt_len, k=1)].sum():.8f}")

## 🌉 Decoder Architecture: The Cross-Attention Bridge

The decoder has THREE sub-layers (vs the encoder's TWO):

```
ENCODER LAYER:                        DECODER LAYER:
═══════════════                       ════════════════════════════

┌──────────────────────┐              ┌──────────────────────────┐
│  1. Self-Attention   │              │ 1. Masked Self-Attention  │
│     (Bidirectional)  │              │    (Causal — past only)   │
│                      │              │                           │
│  2. Feed-Forward     │              │ 2. ⭐ CROSS-ATTENTION ⭐   │
│     Network          │              │    Q from decoder         │
└──────────────────────┘              │    K, V from ENCODER! 🌉  │
                                      │                           │
                                      │ 3. Feed-Forward Network   │
                                      └──────────────────────────┘
```

### Cross-Attention: The Magic Bridge 🌉

This is the most important piece for translation. Here's exactly what happens:

```
The decoder is generating "chat" (the French word for "cat").
It asks the ENCODER: "Hey, which English words should I focus on right now?"

Cross-Attention Calculation:
  Q = decoder hidden state for token being generated  ("chat")
  K = encoder outputs for ALL source tokens           (The, cat, sat, on, the, mat)
  V = encoder outputs for ALL source tokens           (same as K source)

  Attention scores = softmax(Q · K^T / √d_k)

  Result:
  [The=0.05, cat=0.82, sat=0.04, on=0.03, the=0.03, mat=0.03]
                ↑↑↑
          "cat" gets most attention!
          This is how the model knows "chat" ↔ "cat" 🎯

Each new decoder token asks this question for the SAME encoder outputs.
The encoder runs ONCE; the decoder queries it at every generation step.
```

### The Asymmetry That Makes Translation Work

| Step | Input | Description |
|---|---|---|
| Encoder self-attn | Source sentence only | Builds rich source representations |
| Decoder self-attn | Target tokens so far | Builds target context (masked causal) |
| Decoder cross-attn | **Q from target, K/V from source** | Connects target generation to source meaning |


In [ ]:
"""
🌉 Cross-Attention: Queries from Decoder, Keys+Values from Encoder

We'll implement cross-attention and visualize what the decoder
"looks at" in the encoder when generating each French word.
"""

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(2024)  # Different seed for different random weights


def cross_attention(Q_decoder, K_encoder, V_encoder):
    """
    Cross-Attention: decoder queries attend to encoder keys and values.

    Q_decoder:  [tgt_len, d_k]  — queries FROM the decoder
    K_encoder:  [src_len, d_k]  — keys FROM the encoder
    V_encoder:  [src_len, d_v]  — values FROM the encoder

    Returns:
      output:    [tgt_len, d_v] — each decoder position gets a weighted
                                  combination of encoder value vectors
      attn:      [tgt_len, src_len] — attention distribution
    """
    d_k = Q_decoder.shape[-1]
    # [tgt_len, src_len] — how much does each target token attend to each source token?
    scores = Q_decoder @ K_encoder.T / np.sqrt(d_k)

    # Softmax: NO masking needed here (the decoder can see ALL encoder states)
    scores_shifted = scores - scores.max(axis=-1, keepdims=True)
    exp_scores = np.exp(scores_shifted)
    attn = exp_scores / exp_scores.sum(axis=-1, keepdims=True)

    # [tgt_len, d_v] — weighted mix of encoder values
    output = attn @ V_encoder
    return output, attn


# ═══════════════════════════════════════════════════════
# Simulate a translation scenario
# ═══════════════════════════════════════════════════════
src_tokens = ['The', 'cat', 'sat', 'on', 'the', 'mat']
tgt_tokens = ['Le', 'chat', "s'est", 'assis', 'sur', 'le', 'tapis']

src_len = len(src_tokens)
tgt_len = len(tgt_tokens)
d_model = 16  # Larger for more realistic attention patterns

# Simulate encoder output (in practice: output of full encoder stack)
# We slightly bias them to create interpretable attention patterns
encoder_output = np.random.randn(src_len, d_model)

# Simulate decoder Q, K projections (learned weight matrices W_Q, W_K, W_V)
W_Q = np.random.randn(d_model, d_model) * 0.3
W_K = np.random.randn(d_model, d_model) * 0.3
W_V = np.random.randn(d_model, d_model) * 0.3

# Decoder hidden states (output of decoder self-attention step)
decoder_hidden = np.random.randn(tgt_len, d_model)

# Project to Q, K, V
Q_dec = decoder_hidden @ W_Q   # [tgt_len, d_model]
K_enc = encoder_output @ W_K   # [src_len, d_model]
V_enc = encoder_output @ W_V   # [src_len, d_model]

# Run cross-attention!
cross_out, cross_attn = cross_attention(Q_dec, K_enc, V_enc)

print("═" * 60)
print("  🌉 Cross-Attention Output Shapes")
print("═" * 60)
print(f"  Encoder output (K, V source):  {encoder_output.shape} [src_len={src_len}, d_model={d_model}]")
print(f"  Decoder queries (Q source):    {decoder_hidden.shape} [tgt_len={tgt_len}, d_model={d_model}]")
print(f"  Cross-attention output:        {cross_out.shape} [tgt_len={tgt_len}, d_model={d_model}]")
print(f"  Attention weight matrix:       {cross_attn.shape} [tgt_len={tgt_len} × src_len={src_len}]")

print("\n📊 Cross-Attention Matrix (decoder → encoder):")
print(f"   {'':12}", end='')
for s in src_tokens:
    print(f"  {s:6}", end='')
print()
for i, tgt in enumerate(tgt_tokens):
    print(f"   {tgt:12}", end='')
    for j in range(src_len):
        print(f"  {cross_attn[i,j]:.3f}", end='')
    max_j = cross_attn[i].argmax()
    print(f"  ← mostly attends to '{src_tokens[max_j]}'")

print("\n✅ Each row sums to 1.0:")
print(f"   {cross_attn.sum(axis=1).round(6)}")

# ═══════════════════════════════════════════════════════
# Visualization
# ═══════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot 1: Cross-attention heatmap
ax1 = axes[0]
im = ax1.imshow(cross_attn, cmap='YlOrRd', vmin=0, vmax=cross_attn.max(), aspect='auto')
ax1.set_xticks(range(src_len))
ax1.set_yticks(range(tgt_len))
ax1.set_xticklabels(src_tokens, fontsize=12)
ax1.set_yticklabels(tgt_tokens, fontsize=12)
ax1.set_title('🌉 Cross-Attention Weights\n(Decoder → Encoder)', fontsize=14, fontweight='bold')
ax1.set_xlabel('SOURCE Tokens (Encoder Keys/Values)', fontsize=11)
ax1.set_ylabel('TARGET Tokens (Decoder Queries)', fontsize=11)

for i in range(tgt_len):
    for j in range(src_len):
        ax1.text(j, i, f'{cross_attn[i,j]:.2f}',
                 ha='center', va='center', fontsize=10,
                 color='white' if cross_attn[i,j] > 0.25 else 'black')
plt.colorbar(im, ax=ax1)

# Add highlights showing strongest attention per target token
for i in range(tgt_len):
    j = cross_attn[i].argmax()
    rect = plt.Rectangle((j - 0.5, i - 0.5), 1, 1, linewidth=3,
                          edgecolor='blue', facecolor='none')
    ax1.add_patch(rect)

# Plot 2: Diagram showing the data flow
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('🔄 Cross-Attention Data Flow', fontsize=14, fontweight='bold')

# Draw the architecture diagram
boxes = [
    # (x, y, width, height, text, color)
    (1.0, 7.5, 3.5, 1.2, '📥 Encoder Output\n[src_len × d_model]', '#e3f2fd'),
    (1.0, 5.0, 1.5, 1.2, '🔑 W_K\n(Key proj)', '#fff9c4'),
    (2.8, 5.0, 1.5, 1.2, '💎 W_V\n(Value proj)', '#fff9c4'),
    (5.5, 7.5, 3.5, 1.2, '📤 Decoder Hidden State\n[tgt_len × d_model]', '#e8f5e9'),
    (6.5, 5.0, 1.5, 1.2, '❓ W_Q\n(Query proj)', '#fce4ec'),
    (3.5, 2.5, 3.0, 1.2, '⭐ Cross-Attention\nQ·K^T / √d_k + softmax', '#f3e5f5'),
    (3.5, 0.5, 3.0, 1.2, '✨ Output\n[tgt_len × d_model]', '#e8f5e9'),
]

for (x, y, w, h, text, color) in boxes:
    rect = plt.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                               facecolor=color, edgecolor='gray', linewidth=1.5)
    ax2.add_patch(rect)
    ax2.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=8.5,
             fontweight='bold')

# Arrows
arrow_kwargs = dict(arrowstyle='->', lw=2, color='#555555')
ax2.annotate('', xy=(1.75, 6.2), xytext=(2.0, 7.5), arrowprops=arrow_kwargs)
ax2.annotate('', xy=(3.55, 6.2), xytext=(3.0, 7.5), arrowprops=arrow_kwargs)
ax2.annotate('', xy=(7.25, 6.2), xytext=(7.0, 7.5), arrowprops=arrow_kwargs)
# K, V, Q → cross-attention
ax2.annotate('', xy=(4.5, 3.7), xytext=(1.75, 6.2), arrowprops=dict(arrowstyle='->', lw=1.5, color='#1976d2'))
ax2.annotate('', xy=(4.8, 3.7), xytext=(3.55, 6.2), arrowprops=dict(arrowstyle='->', lw=1.5, color='#1976d2'))
ax2.annotate('', xy=(5.2, 3.7), xytext=(7.25, 6.2), arrowprops=dict(arrowstyle='->', lw=1.5, color='#c62828'))
ax2.annotate('', xy=(5.0, 1.7), xytext=(5.0, 2.5), arrowprops=arrow_kwargs)

# Labels on arrows
ax2.text(2.8, 4.5, 'K', fontsize=12, color='#1976d2', fontweight='bold')
ax2.text(3.8, 4.2, 'V', fontsize=12, color='#1976d2', fontweight='bold')
ax2.text(6.3, 4.2, 'Q', fontsize=12, color='#c62828', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🎯 KEY TAKEAWAY:")
print("  Cross-attention has DIFFERENT sources for Q vs K/V!")
print("  Q comes from the DECODER (what am I trying to generate?)")
print("  K and V come from the ENCODER (what does the source mean?)")
print("  This is the bridge between the two languages. 🌉")

## 🏛️ Encoder vs Decoder: The Full Comparison

### Side-by-Side Comparison Table

| Feature | **Encoder** | **Decoder** |
|---|---|---|
| **Attention type** | Bidirectional (full) | Causal (lower-triangular mask) |
| **Input** | Full source sentence | Target tokens generated so far |
| **Attention layers** | 1 per block (self-attention) | 2 per block (self + cross-attention) |
| **Reads encoder?** | N/A (IS the encoder) | Yes! Cross-attention to encoder output |
| **Runs how many times?** | ONCE per input sentence | Once per output token (auto-regressive) |
| **Analogy** | Reading the whole French chapter | Writing the English summary word by word |

### The Full Encoder-Decoder Architecture

```
                    ┌─────────────────────────────────────────────────────────┐
                    │              ENCODER STACK (runs ONCE)                 │
                    │                                                         │
 "The cat sat"      │   [The]  [cat]  [sat]  [on]  [the]  [mat]              │
 (tokenized)   ───► │     │      │      │      │      │      │               │
                    │   ┌─────────────────────────────────────────────┐      │
                    │   │  Word Embeddings + Positional Encoding      │      │
                    │   └─────────────────────────────────────────────┘      │
                    │                         │                               │
                    │            ┌────────────▼────────────┐                 │
                    │            │  Encoder Block ×N        │                 │
                    │            │  ┌──────────────────┐    │                 │
                    │            │  │ Self-Attention   │    │                 │
                    │            │  │ (Bidirectional)  │    │                 │
                    │            │  └──────────┬───────┘    │                 │
                    │            │  ┌──────────▼───────┐    │                 │
                    │            │  │ Feed-Forward     │    │                 │
                    │            │  │ Network          │    │                 │
                    │            │  └──────────────────┘    │                 │
                    │            └────────────┬────────────┘                 │
                    │                         │                               │
                    │              Encoder Output Vectors 🏦                  │
                    └─────────────────────────┬───────────────────────────────┘
                                              │  (passed to ALL decoder blocks)
                                              ▼
                    ┌─────────────────────────────────────────────────────────┐
                    │         DECODER STACK (runs N_output times)            │
                    │                                                         │
 <BOS> + generated  │   [<BOS>]  [Le]  [chat]  ...                           │
 tokens so far  ──► │     │        │       │                                  │
                    │   ┌─────────────────────────────────────────────┐      │
                    │   │  Word Embeddings + Positional Encoding      │      │
                    │   └─────────────────────────────────────────────┘      │
                    │                         │                               │
                    │            ┌────────────▼────────────┐                 │
                    │            │  Decoder Block ×N        │                 │
                    │            │  ┌──────────────────┐    │                 │
                    │            │  │ Masked Self-Attn │    │                 │
                    │            │  │ (Causal only)    │    │                 │
                    │            │  └──────────┬───────┘    │                 │
                    │            │  ┌──────────▼───────┐    │   ◄── Encoder  │
                    │            │  │ Cross-Attention  │◄───────── Output    │
                    │            │  │ Q=decoder        │    │       K,V from  │
                    │            │  │ K,V=encoder 🌉   │    │       encoder   │
                    │            │  └──────────┬───────┘    │                 │
                    │            │  ┌──────────▼───────┐    │                 │
                    │            │  │ Feed-Forward     │    │                 │
                    │            │  │ Network          │    │                 │
                    │            │  └──────────────────┘    │                 │
                    │            └────────────┬────────────┘                 │
                    │                         │                               │
                    │              Linear + Softmax over vocab                │
                    └─────────────────────────┬───────────────────────────────┘
                                              │
                                              ▼
                                   Next token: "Le" → "chat" → ...
```

### Key Architecture Numbers (Google's Original Transformer)

| Hyperparameter | Value | What it controls |
|---|---|---|
| `d_model` | 512 | Embedding dimension throughout |
| `d_ff` | 2048 | Feed-forward hidden layer size |
| `n_heads` | 8 | Number of parallel attention heads |
| `d_k = d_v` | 64 | Per-head dimension (512/8) |
| `N` (layers) | 6 | Number of encoder and decoder blocks each |
| `dropout` | 0.1 | Regularization |
| `vocab_size` | ~37,000 | BPE vocabulary (for WMT EN-DE) |

> 🎯 **Interview fact:** The original Transformer paper ("Attention Is All You Need", Vaswani et al. 2017) used this exact configuration. The base model has ~65M parameters. Modern multilingual models like NLLB-200 scale this up to 54B+ parameters across 200 languages.


## 🧪 Quiz Time! Encoder-Decoder Architecture

Test yourself before the interview! Try answering each question, then reveal.

---

### Question 1: Attention Direction

During encoding of the sentence "The bank on the river bank is closed", can the encoder token for the SECOND "bank" attend to the token "river"?

- A) No — the encoder uses causal masking, so it can only look at past tokens
- B) Yes — the encoder is bidirectional, all tokens can attend to all other tokens
- C) Only if "river" appeared before "bank"
- D) Only in cross-attention, not self-attention

<details>
<summary>🔓 Reveal Answer</summary>

**B) Yes!** The encoder uses BIDIRECTIONAL attention — no masking. Every token can attend to every other token in both directions. This is crucial for disambiguation: the second "bank" can use "river" (which appears right before it) to understand it means "riverbank" not "financial institution". This is impossible with a decoder-only model where causal masking would prevent looking ahead.

</details>

---

### Question 2: Cross-Attention Sources

In cross-attention, where do the Query, Key, and Value vectors come from?

- A) Q from encoder, K and V from decoder
- B) Q, K, V all from encoder
- C) Q from decoder, K and V from encoder
- D) Q from decoder, K from encoder, V from decoder

<details>
<summary>🔓 Reveal Answer</summary>

**C) Q from decoder, K and V from encoder.** This is THE key fact about cross-attention. The decoder asks the question (Q = "what meaning am I trying to generate right now?"), and the encoder provides the answer bank (K = "what topics are in the source?", V = "here's the actual information about each topic"). This asymmetry is what makes the decoder "look up" information from the source language.

> 💡 Memory trick: The decoder is the **customer** (Q = asking), the encoder is the **database** (K = index, V = data).

</details>

---

### Question 3: Why Not GPT?

Your interviewer asks: "Why can't we just use a decoder-only model (like GPT) for translation?" What's the BEST answer?

- A) GPT is too slow for translation
- B) GPT can't generate text, it can only classify
- C) A decoder-only model processes tokens left-to-right, so when generating the first target word it hasn't yet seen the entire source sentence — but translation often requires full-sentence understanding because word order changes between languages
- D) GPT doesn't support multiple languages

<details>
<summary>🔓 Reveal Answer</summary>

**C)** This is the architecturally precise answer. In causal attention, when GPT generates the first output token, it only sees the first source token (in a naive setup). Even with "see-all" setups, you lose the clean separation of fully understanding the source before generating the target. In translation, word order often changes completely (SOV vs SVO vs VSO languages), so generating position 1 of the output requires knowing the full source structure. The encoder solves this by building a complete bidirectional understanding of the source BEFORE any generation begins.

</details>

---

### Question 4: BPE Vocabulary

A model trained on a 50k BPE vocabulary encounters the word "unhappily" — which it has NEVER seen during BPE training. What happens?

- A) The model produces a UNK token — it cannot handle unknown words
- B) The word is split into known subword units like ["un", "happi", "ly"] and processed normally
- C) The model crashes — BPE cannot handle out-of-vocabulary words
- D) The entire sentence is skipped during training

<details>
<summary>🔓 Reveal Answer</summary>

**B)** This is BPE's killer feature! Even for unseen words, BPE breaks them into learned subword units. "unhappily" → ["un", "happi", "ly"] (approximately) — all subwords the model HAS seen. In the worst case, any string can be represented as individual characters (which are always in the vocabulary). This is why modern models essentially never need UNK tokens — there's always a valid tokenization.

</details>

---

### Question 5: Decoder Generation Speed

The encoder runs ONCE for each input sentence. How many times does the decoder run for generating a 50-token translation?

- A) Once — the decoder generates all 50 tokens in parallel
- B) 50 times — once per output token (auto-regressive)
- C) 6 times — once per decoder layer
- D) It depends on the sequence length

<details>
<summary>🔓 Reveal Answer</summary>

**B) 50 times!** This is why inference is slow — each token requires a full decoder forward pass. The encoder runs once and caches its output; the decoder then generates token-by-token, each time receiving the full encoder output via cross-attention. This auto-regressive generation is the bottleneck for translation latency. Solutions: speculative decoding (guess multiple tokens at once), non-autoregressive decoders, or distillation into smaller models. The encoder's output is also cached (stored as K,V pairs) so you don't recompute it — this is called the **KV cache**.

</details>
